# 📥 Stage-1: Dense Similarity Pre-computation (Retriever)

This notebook pre-computes the cosine similarity scores between all queries and document candidates using one of the following models:
- [] Qwen/Qwen3-Embedding-8B
- [] BAAI/bge-m3
- [] jinaai/jina-embeddings-v4
- [] jinaai/jina-embeddings-v3

Calculated dense scores are stored in `..outputs/cache/dense_scores`

### 🧭 Why Pre-compute?
Calculating high-dimensional embeddings and computing pairwise similarities for all 25+ candidates dynamically during Stage-2 reranking can consume valuable VRAM and slow down inference. Pre-computing similarity scores allows us to perform selective **Top-K pre-filtering (filtering out the noise, keeping only the Top-8 candidates)**. This reduces the prompt length by **33%** and prevents GPU CUDA Out of Memory (OOM) errors during cross-encoder reranking.

In [2]:
# === IMPORT LIBRARIES ===
import sys
import os
import json
import ctypes
from pathlib import Path
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

In [3]:
# === RESOLVE PATHS ===

# Setup path relative to parent directory early so imports from src work
base_dir = Path("..").resolve()
sys.path.append(str(base_dir))

from src.data.loader import load_jsonl
from src.retrievers.dense import DenseRetriever

# Force CUDA library paths for NVRTC and bitsandbytes compatibility
cuda_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/"
if cuda_path not in os.environ.get("LD_LIBRARY_PATH", ""):
	os.environ["LD_LIBRARY_PATH"] = cuda_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Preload CUDA dependencies to ensure bitsandbytes initializes correctly
try:
	ctypes.CDLL(os.path.join(cuda_path, "libnvJitLink.so.13"))
	ctypes.CDLL(os.path.join(cuda_path, "libnvrtc.so.13"))
	ctypes.CDLL(os.path.join(cuda_path, "libnvrtc-builtins.so.13.0"))
	print("✅ CUDA libraries pre-loaded successfully!")
except Exception as e:
	print(f"⚠️ Pre-loading CUDA libraries failed: {e}")
from src.retrievers.dense import DenseRetriever

# Setup path relative to parent directory
base_dir = Path("..").resolve()
sys.path.append(str(base_dir))

# Force CUDA library paths for NVRTC and bitsandbytes compatibility
cuda_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/"
if cuda_path not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = cuda_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Preload CUDA dependencies to ensure bitsandbytes initializes correctly
try:
    ctypes.CDLL(os.path.join(cuda_path, "libnvJitLink.so.13"))
    ctypes.CDLL(os.path.join(cuda_path, "libnvrtc.so.13"))
    ctypes.CDLL(os.path.join(cuda_path, "libnvrtc-builtins.so.13.0"))
    print("✅ CUDA libraries pre-loaded successfully!")
except Exception as e:
    print(f"⚠️ Pre-loading CUDA libraries failed: {e}")

✅ CUDA libraries pre-loaded successfully!
✅ CUDA libraries pre-loaded successfully!


In [ ]:
# === EXPERIMENT CONFIGURATION ===
data_dir = base_dir / "data"
cache_dir = base_dir / "outputs/cache"
cache_dir.mkdir(parents=True, exist_ok=True)

os.environ["DISABLE_VLM"] = "1" # 0: no, 1: yes
os.environ["VLM_CAPTIONS_FILE"] = "qwen2-vl-7b-instruct_image_captions.json"

print("🚀 Loading Dense Retriever Model...")
model_name = "BAAI/bge-large-en-v1.5"
model_subName = "bge-large-en-v1.5"
retriever = DenseRetriever(model_name=model_name, load_in_4bit=False)
#retriever = SentenceTransformer(model_name, trust_remote_code=True)
print("🔥 Model successfully loaded!")

In [ ]:
datasets = {
    "train": data_dir / "train.jsonl",
    "test": data_dir / "test.jsonl"
}

for name, path in datasets.items():
    if not path.exists():
        print(f"⚠️ Dataset not found at {path}. Skipping.")
        continue
        
    print(f"\n⚙️ Generating/Resuming dense scores cache for {name} set...")
    data = load_jsonl(str(path))
    
    out_file = cache_dir / f"{model_subName}_dense_scores_{name}.json"
    
    # Load existing progress if file exists
    cache_dict = {}
    if out_file.exists():
        try:
            with open(out_file, "r") as f:
                cache_dict = json.load(f)
            print(f"🔄 Resuming: Found {len(cache_dict)} already scored samples in cache.")
        except Exception as e:
            print(f"⚠️ Failed to load existing cache: {e}. Starting fresh.")
    
    # Filter remaining samples to process
    to_process = [sample for sample in data if str(sample.q_id) not in cache_dict]
    print(f"⏳ Samples remaining to process: {len(to_process)}")
    
    if not to_process:
        print(f"✅ All {name} samples already fully scored!")
        continue
        
    # Score remaining samples
    count = 0
    save_interval = 50
    
    for sample in tqdm(to_process, desc=f"Scoring {name}"):
        preds = retriever.retrieve(sample, top_k=100)
        # Save ONLY the ordered quote_ids (their order is their similarity rank)
        cache_dict[str(sample.q_id)] = [p[0] for p in preds]
        
        count += 1
        if count >= save_interval:
            with open(out_file, "w") as f:
                json.dump(cache_dict, f, indent=2)
            count = 0
            
    # Final save for the dataset
    with open(out_file, "w") as f:
        json.dump(cache_dict, f, indent=2)
    print(f"✅ Cache successfully saved/updated at: {out_file}")
    
print("\n=== 🎉 DENSE PRE-COMPUTATION COMPLETED! ===")